# Tugas Praktik SVM

Anggota :

1.   Muhammad Zayyan Achnaf (24523147)
2.   Muhammad Rifki Apreliant (24523097)



# Tema : Dataset Gaya Hidup dan Stress Siswa

# 1. Klasifikasi dan Menjumlahkan missing value

In [ ]:
import pandas as pd

df = pd.read_csv('/content/sample_data/student-lifestyle-and-stress-dataset.csv')

print("--- 5 Baris Pertama ---")
display(df.head())

print("\n--- Jumlah Missing Value Sebelum Preprocessing ---")
print(df.isnull().sum())

--- 5 Baris Pertama ---


,Student_Type,Sleep_Hours,Study_Hours,Social_Media_Hours,Attendance,Exam_Pressure,Family_Support,Month,Stress_Level
0,school,6.868702,1.711722,3.176942,NaN,8.0,7.0,2.0,1
1,school,8.519088,3.251084,3.880787,93.978465,6.0,4.0,3.0,1
2,college,4.498770,6.306885,2.936172,64.421253,7.0,1.0,12.0,1
3,school,8.591223,2.384922,5.222832,81.868960,2.0,7.0,7.0,0
4,college,5.329293,9.345179,7.815869,85.847982,5.0,6.0,10.0,1



--- Jumlah Missing Value Sebelum Preprocessing ---
Student_Type          1252
Sleep_Hours           1333
Study_Hours           1277
Social_Media_Hours    1312
Attendance            1305
Exam_Pressure         1270
Family_Support        1291
Month                 1314
Stress_Level             0
dtype: int64


 Kami sepakat untuk memilih Dataset Klasifikasi dari Kaggle dengan temanya "Dataset Gaya Hidup dan Stress Mahasiswa",

 Klasifikasi target untuk mengetahui apakah target stress dengan indikator adalah seperti berikut (0 = Tidak Stres, 1 = Stres)

 Dengan Kesimpulan: "Jam tidur mahasiswa" memiliki missing value tertinggi (1333) dan "Tingkat stress mahasiswa" adalah missing value paling rendah (0) atau bisa dikatakan tidak ada sama sekali.

# 2. Preprocessing Data

In [ ]:
from sklearn.preprocessing import StandardScaler

df['Student_Type'] = df['Student_Type'].fillna(df['Student_Type'].mode()[0])

kolom_numerik = ['Sleep_Hours', 'Study_Hours', 'Social_Media_Hours',
                 'Attendance', 'Exam_Pressure', 'Family_Support', 'Month']
for col in kolom_numerik:
    df[col] = df[col].fillna(df[col].median())

df['Student_Type'] = df['Student_Type'].map({'school': 0, 'college': 1})
df['Student_Type'] = df['Student_Type'].fillna(df['Student_Type'].mode()[0])

X = df.drop('Stress_Level', axis=1)
y = df['Stress_Level']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("--- 5 Baris Pertama Setelah Preprocessing & Scaling ---")
display(X_scaled_df.head())

--- 5 Baris Pertama Setelah Preprocessing & Scaling ---


,Student_Type,Sleep_Hours,Study_Hours,Social_Media_Hours,Attendance,Exam_Pressure,Family_Support,Month
0,-1.79866,0.276812,-1.363288,-0.202945,0.054309,1.134350,0.547678,-1.329185
1,-1.79866,1.403762,-0.658311,0.177108,1.079454,0.277806,-0.831295,-1.032172
2,0.55597,-1.341473,0.741145,-0.332953,-1.438132,0.706078,-2.210269,1.640943
3,-1.79866,1.453018,-1.054985,0.901768,0.048006,-1.435283,0.547678,0.155879
4,0.55597,-0.774359,2.132583,2.301922,0.386926,-0.150466,0.088020,1.046918


Masuk ke tahap Preprocessing data kami mulai menangani missing value, 'Student Type' kami jadikan sebagai kolom kategorikal nilai MODUS yang paling sering muncul, lalu kolom angka seperti 'Sleep Hours', dll sebagai MEDIAN, MEAN tidak digunakan karna akan merusak Outlier jika halnya terjadi mahasiswa yang tidur tidak wajar, maka dari itu MEDIAN akan lebih stabil

Selanjutnya kita melakukan Encoding (proses mapping untuk mengubah data format non numerik menjadi numerik), beserta memisahkan fitur - fitur agar sesuai.

Lanjut ke Scaling Data untuk menyamakan rentang angka nilai fitur - fitur, 'Student Type' memiliki 0 - 1, sedangkan 'Sleep Hours' adalah 0 - 100 dan 'Kehadiran' belum dapat dipastikan (sekitar puluhan)

Maka dari itu menggunakan Standard Scaler, mencakup semua kolom dan mengubah skala didalamnya menjadi seimbang.

Dengan hasilnya,

Angka 0: Jika hasilnya mendekati 0, berarti data mahasiswa tersebut sama persis dengan rata-rata mahasiswa lainnya di dataset.

Angka Positif (Contoh: 1.25): Berarti nilai mahasiswa tersebut berada di atas rata-rata.

Angka Negatif (Contoh: -0.85): Berarti nilainya berada di bawah rata-rata.

# 3. Split Data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.2, random_state=42)

print(f"Jumlah Total Data: {len(df)}")
print(f"Jumlah Data Latih (Train): {len(X_train)}")
print(f"Jumlah Data Uji (Test): {len(X_test)}")

Jumlah Total Data: 25500
Jumlah Data Latih (Train): 20400
Jumlah Data Uji (Test): 5100


Kenapa perlunya latihan dan uji data?

Disitu kita dapat memverifikasi apakah data yang disimpan komputer sudah ter-*filter* atau belum, karena data yang sudah teruji dapat dipastikan lolos seleksi.

Terdapat jumlah total data-nya : 25500, Data Latih : 20400 dana Data Uji : 5100

Maka dari itu data yang diproses selanjutnya adalah 5100 data.

# 4. Modeling SVM

In [ ]:
# Import library yang dibutuhkan untuk modeling, evaluasi, dan visualisasi
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# LANGKAH 4: MODELING SVM
# ==========================================
# Pilihan Kernel: RBF (Radial Basis Function)
# Alasan: Hubungan antara faktor gaya hidup (seperti jam tidur, jam belajar)
# dengan stres biasanya kompleks dan tidak berbentuk garis lurus (non-linear).
# Kernel RBF sangat handal menangani pola data melengkung/kompleks seperti ini
# tanpa kita harus mengubah fitur datanya secara manual.

print("Sedang melatih model SVM... (Ini mungkin memakan waktu beberapa detik)\n")
svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train, y_train)

Sedang melatih model SVM... (Ini mungkin memakan waktu beberapa detik)



SVC(random_state=42)

In [ ]:
# ==========================================
# LANGKAH 5: EVALUASI MODEL
# ==========================================
# Melakukan prediksi pada data Uji (test)
y_pred_test = svm_model.predict(X_test)

# Melakukan prediksi pada data Latih (train) untuk mengecek overfitting nanti
y_pred_train = svm_model.predict(X_train)

# Menghitung Metrik
acc = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test)
rec = recall_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)

print("--- HASIL EVALUASI MODEL SVM ---")
print(f"Accuracy  : {acc:.4f} (Seberapa banyak tebakan benar dari keseluruhan data)")
print(f"Precision : {prec:.4f} (Dari yang ditebak 'Stres', berapa banyak yang Beneran Stres)")
print(f"Recall    : {rec:.4f} (Dari seluruh orang yang Beneran Stres, berapa yang berhasil ditebak model)")
print(f"F1 Score  : {f1:.4f} (Keseimbangan antara Precision dan Recall)\n")

# --- Analisis Overfitting vs Underfitting ---
acc_train = accuracy_score(y_train, y_pred_train)
print("--- ANALISIS OVERFITTING / UNDERFITTING ---")
print(f"Akurasi pada Data Latih (Train) : {acc_train:.4f}")
print(f"Akurasi pada Data Uji (Test)    : {acc:.4f}")

# Logika sederhana untuk menyimpulkan kondisi model
selisih = acc_train - acc
if selisih > 0.05:
    print("Kesimpulan: Model cenderung OVERFITTING.")
    print("Alasan: Model terlalu 'menghafal' data latih sehingga akurasinya tinggi di sana, tapi performanya turun signifikan saat bertemu data baru (data uji).")
elif acc_train < 0.65 and acc < 0.65:
    print("Kesimpulan: Model cenderung UNDERFITTING.")
    print("Alasan: Model gagal mempelajari pola data dengan baik. Akurasinya rendah baik di data latih maupun data uji.")
else:
    print("Kesimpulan: Model tergolong GOOD FIT (Optimal).")
    print("Alasan: Performa model stabil dan seimbang. Akurasi di data latih dan data uji mirip/berdekatan.")

In [ ]:
# ==========================================
# LANGKAH 6: VISUALISASI
# ==========================================
# Kita buat canvas dengan 3 grafik berjajar ke samping
plt.figure(figsize=(18, 5))

# 1. Visualisasi Distribusi Kelas Target (Imbalanced/Balanced Check)
plt.subplot(1, 3, 1)
# Kita pakai DataFrame 'df' (data awal) karena targetnya belum diubah
sns.countplot(data=df, x='Stress_Level', palette='Set2')
plt.title('1. Distribusi Kelas (Stres vs Tidak)')
plt.xlabel('Tingkat Stres (0=Tidak, 1=Ya)')
plt.ylabel('Jumlah Kasus')

# 2. Visualisasi Confusion Matrix
plt.subplot(1, 3, 2)
cm = confusion_matrix(y_test, y_pred_test)
# Menampilkan matrix dengan warna heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Prediksi Tidak', 'Prediksi Stres'],
            yticklabels=['Asli Tidak', 'Asli Stres'])
plt.title('2. Confusion Matrix (Hasil Tebakan)')
plt.xlabel('Tebakan Model')
plt.ylabel('Data Sebenarnya')

# 3. Visualisasi Fitur Sederhana (Hubungan Jam Tidur & Stres)
# Menggunakan Boxplot untuk melihat apakah perbedaan jam tidur mempengaruhi stres
plt.subplot(1, 3, 3)
sns.boxplot(data=df, x='Stress_Level', y='Sleep_Hours', palette='Set2')
plt.title('3. Hubungan Jam Tidur vs Tingkat Stres')
plt.xlabel('Tingkat Stres (0=Tidak, 1=Ya)')
plt.ylabel('Jam Tidur (Hours)')

# Menyesuaikan jarak antar grafik agar rapi, lalu menampilkannya
plt.tight_layout()
plt.show()

Pembuatan Modeling SVM,

Untuk membuat wadah otak nya perlu SVC lalu kita import sekaligus kita *declare variable* untuk menghitung Akurasi, Presisi, Recall, F1 Score

Kami mengambil jenis kernel RBF (Radial Basis Function), batas antar mahasiswa stress dan yang tidak itu rumit dan tidak bisa ditarik dengan satu garis lurus saja (non-linear). Adanya Kernel RBF membuat SVM untuk menarik garis pembatas yang melengkung dan fleksibel, mengikuti pola sebaran data yang kompleks.

# 5. Evaluasi Model

In [ ]:
# ==========================================
# LANGKAH 5: EVALUASI MODEL
# ==========================================
# Melakukan prediksi pada data Uji (test)
y_pred_test = svm_model.predict(X_test)

# Melakukan prediksi pada data Latih (train) untuk mengecek overfitting nanti
y_pred_train = svm_model.predict(X_train)

# Menghitung Metrik
acc = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test)
rec = recall_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)

print("--- HASIL EVALUASI MODEL SVM ---")
print(f"Accuracy  : {acc:.4f} (Seberapa banyak tebakan benar dari keseluruhan data)")
print(f"Precision : {prec:.4f} (Dari yang ditebak 'Stres', berapa banyak yang Beneran Stres)")
print(f"Recall    : {rec:.4f} (Dari seluruh orang yang Beneran Stres, berapa yang berhasil ditebak model)")
print(f"F1 Score  : {f1:.4f} (Keseimbangan antara Precision dan Recall)\n")

# --- Analisis Overfitting vs Underfitting ---
acc_train = accuracy_score(y_train, y_pred_train)
print("--- ANALISIS OVERFITTING / UNDERFITTING ---")
print(f"Akurasi pada Data Latih (Train) : {acc_train:.4f}")
print(f"Akurasi pada Data Uji (Test)    : {acc:.4f}")

# Logika sederhana untuk menyimpulkan kondisi model
selisih = acc_train - acc
if selisih > 0.05:
    print("Kesimpulan: Model cenderung OVERFITTING.")
    print("Alasan: Model terlalu 'menghafal' data latih sehingga akurasinya tinggi di sana, tapi performanya turun signifikan saat bertemu data baru (data uji).")
elif acc_train < 0.65 and acc < 0.65:
    print("Kesimpulan: Model cenderung UNDERFITTING.")
    print("Alasan: Model gagal mempelajari pola data dengan baik. Akurasinya rendah baik di data latih maupun data uji.")
else:
    print("Kesimpulan: Model tergolong GOOD FIT (Optimal).")
    print("Alasan: Performa model stabil dan seimbang. Akurasi di data latih dan data uji mirip/berdekatan.")

KeyboardInterrupt: 

# 6. Visualisasi

In [ ]:
# ==========================================
# LANGKAH 6: VISUALISASI
# ==========================================
# Kita buat canvas dengan 3 grafik berjajar ke samping
plt.figure(figsize=(18, 5))

# 1. Visualisasi Distribusi Kelas Target (Imbalanced/Balanced Check)
plt.subplot(1, 3, 1)
# Kita pakai DataFrame 'df' (data awal) karena targetnya belum diubah
sns.countplot(data=df, x='Stress_Level', palette='Set2')
plt.title('1. Distribusi Kelas (Stres vs Tidak)')
plt.xlabel('Tingkat Stres (0=Tidak, 1=Ya)')
plt.ylabel('Jumlah Kasus')

# 2. Visualisasi Confusion Matrix
plt.subplot(1, 3, 2)
cm = confusion_matrix(y_test, y_pred_test)
# Menampilkan matrix dengan warna heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Prediksi Tidak', 'Prediksi Stres'],
            yticklabels=['Asli Tidak', 'Asli Stres'])
plt.title('2. Confusion Matrix (Hasil Tebakan)')
plt.xlabel('Tebakan Model')
plt.ylabel('Data Sebenarnya')

# 3. Visualisasi Fitur Sederhana (Hubungan Jam Tidur & Stres)
# Menggunakan Boxplot untuk melihat apakah perbedaan jam tidur mempengaruhi stres
plt.subplot(1, 3, 3)
sns.boxplot(data=df, x='Stress_Level', y='Sleep_Hours', palette='Set2')
plt.title('3. Hubungan Jam Tidur vs Tingkat Stres')
plt.xlabel('Tingkat Stres (0=Tidak, 1=Ya)')
plt.ylabel('Jam Tidur (Hours)')

# Menyesuaikan jarak antar grafik agar rapi, lalu menampilkannya
plt.tight_layout()
plt.show()

## Kesimpulan Akhir Analisis Data Gaya Hidup dan Stres Siswa

Dari analisis yang telah dilakukan pada dataset "Gaya Hidup dan Stres Siswa", beberapa poin penting dapat disimpulkan:

1.  **Penanganan Missing Value**: Data awal memiliki sejumlah missing value pada beberapa kolom seperti `Student_Type`, `Sleep_Hours`, `Study_Hours`, dll. Kami telah mengisi missing value ini dengan metode yang sesuai: menggunakan `mode` untuk kolom kategorikal (`Student_Type`) dan `median` untuk kolom numerik. Penggunaan median pada kolom numerik membantu mencegah distorsi data akibat outlier, menjaga keakuratan distribusi data.

2.  **Preprocessing Data**: Kolom `Student_Type` di-encode menjadi nilai numerik (0 dan 1) untuk keperluan pemodelan. Selanjutnya, data fitur (`X`) di-scaling menggunakan `StandardScaler`. Proses scaling ini penting untuk menyamakan rentang nilai antar fitur, memastikan bahwa semua fitur berkontribusi secara proporsional pada model dan mencegah fitur dengan skala besar mendominasi proses pembelajaran.

3.  **Pembagian Data**: Data telah dibagi menjadi data latih (80%) dan data uji (20%). Pembagian ini krusial untuk mengevaluasi performa model pada data yang belum pernah dilihat sebelumnya, sehingga memberikan indikasi yang lebih akurat tentang kemampuan generalisasi model.

4.  **Modeling SVM**: Model Support Vector Machine (SVM) dengan kernel Radial Basis Function (RBF) digunakan untuk klasifikasi. Pemilihan kernel RBF didasarkan pada asumsi bahwa hubungan antara faktor gaya hidup dan tingkat stres bersifat non-linear dan kompleks, di mana RBF efektif dalam menemukan batas keputusan yang melengkung dan fleksibel.

5.  **Evaluasi Model**: Hasil evaluasi model SVM memberikan metrik seperti Akurasi, Presisi, Recall, dan F1 Score. Penting juga untuk menganalisis perbedaan akurasi antara data latih dan data uji untuk mendeteksi potensi *overfitting* atau *underfitting*. Model yang baik akan menunjukkan keseimbangan performa antara data latih dan data uji.

6.  **Visualisasi**: Visualisasi membantu dalam memahami distribusi kelas target, efektivitas model melalui Confusion Matrix, dan hubungan antara fitur-fitur tertentu (misalnya, jam tidur) dengan tingkat stres. Ini memberikan wawasan visual yang mendukung temuan numerik.

Secara keseluruhan, tahapan-tahapan ini memberikan kerangka kerja yang komprehensif untuk membangun dan mengevaluasi model klasifikasi yang memprediksi tingkat stres mahasiswa berdasarkan gaya hidup mereka.